# tensor-item-scalar — ex5: tensor → Python control flow

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-item-scalar`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `.item()` patterns that ramp from 0-D extract → single-elem extract → dtype preservation → `.item()` vs `.tolist()` → tensor → Python control flow. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-item-scalar`**, which bridges to the bank subtopic `Numpy: Core array literacy` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-item-scalar"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## `.item()` — quick refresher

**What it does.** Extracts a Python scalar (float / int / bool) from a tensor that has exactly one element. Bridges tensor space to Python space.

**Requirements.** `x.numel() == 1`. Shape doesn't matter — `(1,)`, `(1, 1)`, `()` all work as long as there's exactly one element. Multi-element tensors raise `RuntimeError`.

**Dtype mapping:**
- `float32` / `float64` → Python `float`
- `int32` / `int64` → Python `int`
- `bool` → Python `bool`

**When to use:** logging losses, control-flow conditions, returning counts to callers, dict / set keys (tensors aren't hashable).

**When NOT to use:** inside tight inner loops on GPU — every `.item()` is a host-device sync that blocks the kernel queue.

**For more elements:** `.tolist()` returns a (nested) Python list of any shape.

### Exercise 5 — tensor → Python control flow

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize tensor reduction + `.item()` to bridge tensor space to Python space, returning a Python-typed result usable in `if` / `range` / logging.
> Keywords: control-flow, reduce-then-item, logging-loop, multi-kc
> ```

**KCs targeted:** `item-from-zero-dim`, `item-dtype-preservation`, `item-for-python-control-flow`

Implement `ex5_count_above(x, threshold)`. The canonical 'tensor → Python scalar for control flow' pattern.

Given a 2-D tensor `x` of shape `(B, D)` and a scalar `threshold`, count how many rows have L2 norm strictly greater than `threshold`. Return a plain Python `int` (NOT a 0-D tensor).

Steps:
1. Compute per-row L2 norms — shape `(B,)`.
2. Build a bool mask `norms > threshold`.
3. Sum the mask to get a 0-D `int64` tensor.
4. `.item()` to get a Python `int`.

Why a Python int? So callers can use the count in `if count > 0:` without `.item()` boilerplate, or pass it to `range(...)`.

> ⚠️ **Integrative exercise.** Combines 3 KCs (0-D-extract, dtype preservation, tensor→Python control flow bridge). Empirical work (Lohr et al. ITiCSE 2025) shows 3-concept exercises drop to ~40% solvability — expect a step up vs Exercises 1-4.

In [ ]:
def ex5_count_above(x: Tensor, threshold: float) -> int:
    """Count rows with L2 norm > threshold, return a Python int."""
    raise NotImplementedError()


def _test_ex5():
    x = t.tensor([
        [3.0, 4.0],   # norm 5
        [0.0, 0.0],   # norm 0
        [1.0, 0.0],   # norm 1
        [6.0, 8.0],   # norm 10
    ])
    # threshold = 0.5: rows 0, 2, 3 qualify (norms 5, 1, 10).
    count = ex5_count_above(x, threshold=0.5)
    assert isinstance(count, int), f'expected int, got {type(count).__name__}'
    assert count == 3, f'expected 3, got {count}'

    # threshold = 4.0: rows 0, 3 qualify (norms 5, 10).
    assert ex5_count_above(x, threshold=4.0) == 2

    # threshold = 100.0: no rows qualify.
    zero_count = ex5_count_above(x, threshold=100.0)
    assert isinstance(zero_count, int) and zero_count == 0

    # Must be a Python int — usable in range().
    consumed = list(range(ex5_count_above(x, threshold=0.5)))
    assert consumed == [0, 1, 2], 'result must be usable in range()'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_count_above(x: Tensor, threshold: float) -> int:
    norms = x.pow(2).sum(dim=1).sqrt()
    mask = norms > threshold
    return mask.sum().item()
```

**Why a Python int and not a 0-D tensor?** Callers that use the result as a loop bound, condition, or array length need a Python scalar. PyTorch is fine with most operator overloads, but `range(t.tensor(5))` raises in some versions, and a 0-D tensor stored in a dict key won't hash. `.item()` is the unambiguous bridge.

**Cost.** `.item()` is a host-device sync if `x` is on GPU — every call blocks the kernel queue. Inside a tight training loop you should minimise `.item()` calls (cache the loss tensor, `.item()` only when logging) but for one-off control flow it's fine.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()